In [3]:
"""Data mapping module for standardizing and transforming suicide study dataset.
Handles data from both 2023 and 2013-2022 periods."""

import pandas as pd
import numpy as np

import sys
from pathlib import Path
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

WORKSPACE_PATH = os.getenv("WORKSPACE_PATH")

# Add the parent directory to the system path
sys.path.append(str(WORKSPACE_PATH))

from src.config.config import DATA_DIR

In [ ]:
# ================================================================================
# MAPPING DICTIONARIES
# ================================================================================

# Column name mappings
COLUMN_MAPPINGS = {
    "ID samobójcy": "ID",
    "Data raportu [RRRRMM]": "Date",
    "Przedział wiekowy": "AgeGroup",
    "Płeć": "Gender",
    "Stan cywilny": "Marital",
    "Wykształcenie": "Education",
    "Informacje o pracy i nauce": "WorkInfo",
    "Źródło utrzymania": "Income",
    "Czy samobójstwo zakończyło się zgonem": "Fatal",
    "Miejsce zamachu": "Place",
    "Sposób popełnienia": "Method",
    "Stan świadomości": "Substance",
    "Informacje dotyczące leczenia z powodu alkoholizmu/narkomanii": "AbuseInfo2",
    "Powód zamachu": "Context1",
    "Powód zamachu 2": "Context2",
    "Powód zamachu 3": "Context3",
    "Powód zamachu 4": "Context4",
    "Informacje o używaniu substancji": "AbuseInfo1",
}

# Value mappings for each column
VALUE_MAPPINGS = {
    "AgeGroup": {
        0: np.nan,
        1: "07_12",
        2: "13_18",
        3: "19_24",
        4: "25_29",
        5: "30_34",
        6: "35_39",
        7: "40_44",
        8: "45_49",
        9: "50_54",
        10: "55_59",
        11: "60_64",
        12: "65_69",
        13: "70_74",
        14: "75_79",
        15: "80_84",
        16: "85",
    },
    "Gender": {0: np.nan, 1: "F", 2: "M"},
    "Marital": {
        0: np.nan,
        1: "Single",
        2: "Cohabitant",
        3: "Married",
        4: "Separated",
        5: "Divorced",
        6: "Widowed",
    },
    "Education": {
        0: np.nan,
        1: "PrePrimary",
        2: "Primary",
        3: "Secondary",
        4: "Vocational",
        5: "Secondary",
        6: "Secondary",
        7: "Higher",
    },
    "WorkInfo": {
        0: np.nan,
        1: "Student",
        2: "Student",
        3: "Employed",
        4: "Employed",
        5: "Agriculturalist",
        6: "Employed",
        7: "Employed",
        8: "Employed",
        9: "Unemployed",
    },
    "Income": {0: np.nan, 1: "Dependent", 2: "Steady", 3: "Benefits", 4: "NoSteady"},
    "Fatal": {0: np.nan, 1.0: 1, 2.0: 0},
    "Place": {
        0: np.nan,
        1: "Road",
        2: "UtilitySpaces",
        3: "House",
        4: "Railway",
        5: "Forest",
        6: "House",
        7: "WaterRes",
        8: "Work",
        9: "Institution",
        10: "Isolation",
        11: "PoliceArmy",
        12: "Institution",
        13: "School",
        14: "PoliceArmy",
        15: "Other",
    },
    "Method": {
        0: np.nan,
        1: "Vehicle",
        2: "Jumping",
        3: "Hanging",
        4: "SelfHarm",
        5: "Schooting",
        6: "SelfHarm",
        7: "SelfHarm",
        8: "Drugs",
        9: "Poisoning",
        10: "Gas",
        11: "Drugs",
        12: "Poisoning",
        13: "Drugs",
        14: "Drugs",
        15: "Drowning",
        16: "SelfHarm",
        17: "Other",
        18: "Other",
    },
    "Substance": {
        0: np.nan,
        1: "Sober",
        2: "Alco",
        3: "OtherSub",
        4: "OtherSub",
        5: "OtherSub",
        6: "OtherSub",
        7: "AlcoOtherSub",
        8: "AlcoOtherSub",
        9: "AlcoOtherSub",
        10: "AlcoOtherSub",
        11: "AlcoOtherSub",
        12: "OtherSub",
        13: "OtherSub",
        14: "OtherSub",
        15: "OtherSub",
        16: "OtherSub",
        17: "AlcoOtherSub",
        18: "AlcoOtherSub",
    },
    "Context": {
        0: np.nan,
        1: "HeartBreak",
        2: "MentalHealth",
        3: "FamilyConflict",
        4: "HealthLoss",
        5: "Finances",
        6: "Finances",
        7: "HealthLoss",
        8: "SchoolWork",
        9: "CloseDeath",
        10: "Crime",
        11: "Disability",
        12: "Other",
        13: "HealthLoss",
        14: "HealthLoss",
        15: "SchoolWork",
        16: "Finances",
        17: "SchoolWork",
        18: "Other",
    },
    "AbuseInfo1": {
        0: np.nan,
        1: "Alco",
        2: "OtherSub",
        3: "OtherSub",
        4: "AlcoOtherSub",
        5: "OtherSub",
        6: "AlcoOtherSub",
        7: "AlcoOtherSub",
    },
    "AbuseInfo2": {0: np.nan, 1: "Alco", 2: "OtherSub", 3: "AlcoOtherSub"},
}

In [ ]:
# ================================================================================
# Functions
# ================================================================================

In [129]:
def map_column(df: pd.DataFrame, old_col: str, new_col: str) -> pd.DataFrame:
    """Rename a column in the DataFrame.

    Args:
        df: Input DataFrame
        old_col: Original column name
        new_col: New column name

    Returns:
        DataFrame with renamed column
    """
    if old_col in df.columns:
        df.rename(columns={old_col: new_col}, inplace=True)
    return df

In [130]:
def map_columns(df: pd.DataFrame, column_mappings: dict) -> pd.DataFrame:
    """Rename multiple columns in the DataFrame based on a mapping dictionary.

    Args:
        df: Input DataFrame
        column_mappings: Dictionary mapping old column names to new ones

    Returns:
        DataFrame with renamed columns
    """
    for old_col, new_col in column_mappings.items():
        df = map_column(df, old_col, new_col)
    return df

In [131]:
def load_data(file_path: str, is_excel: bool = True):
    # Load dataset from the specified file path.
    if is_excel:
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path, low_memory=False)
    return df

In [132]:
def map_features(
    df: pd.DataFrame, column_mappings: dict, value_mappings: dict
) -> pd.DataFrame:
    """Map multiple features using provided column and value mappings.

    Args:
        df: Input DataFrame
        column_mappings: Dictionary mapping old column names to new ones
        value_mappings: Dictionary of dictionaries with value mappings for each column

    Returns:
        DataFrame with all specified columns mapped
    """
    for old_col, new_col in column_mappings.items():
        if new_col in value_mappings:
            df[new_col] = df[new_col].map(value_mappings[new_col])

    # Replace values in Context columns
    context_columns = ["Context1", "Context2", "Context3", "Context4"]
    if "Context" in value_mappings:
        for context_col in context_columns:
            if context_col in df.columns:
                df[context_col] = df[context_col].map(value_mappings["Context"])

    return df

In [141]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    # Clean ID column
    if "ID" in df.columns:
        df = df[~df["ID"].isna() & (df["ID"].str.strip() != "")]

    # Process dates if present
    if "Date" in df.columns:
        # Convert 'YYYYMM' format to datetime
        df["Date"] = pd.to_datetime(
            df["Date"], format="%Y%m", errors="coerce"
        )  # Coerce invalid formats to NaT
        # Check for any remaining NaT values and convert them from standard date format
        df["Date"] = pd.to_datetime(
            df["Date"], errors="coerce"
        )  # Convert valid date strings to datetime

        # Create year and month columns
        df["DateY"] = df["Date"].dt.strftime("%Y")
        df["DateM"] = df["Date"].dt.strftime("%m")
        df["Date"] = df["DateM"] + "." + df["DateY"]

    # Find context columns dynamically
    context_columns = [col for col in df.columns if col.startswith("Context")]

    # Extract unique context values
    context_values = set()
    for col in context_columns:
        if col in df.columns:
            context_values.update(df[col].dropna().unique())

    # Create new binary columns for each unique context value
    for value in context_values:
        column_name = f"Context_{value}"
        df[column_name] = df[context_columns].apply(
            lambda row: 1 if value in row.values else 0, axis=1
        )

    # Drop original context columns
    df.drop(columns=context_columns, inplace=True, errors="ignore")

    # Merge AbuseInfo columns
    abuse_info_columns = [col for col in df.columns if col.startswith("AbuseInfo")]
    if abuse_info_columns:
        if len(abuse_info_columns) == 1:
            df["AbuseInfo"] = df[
                abuse_info_columns[0]
            ]  # Directly assign the single column
        else:
            df["AbuseInfo"] = (
                df[abuse_info_columns].bfill(axis=1).iloc[:, 0]
            )  # Use backfill to fill NaNs
        df.drop(columns=abuse_info_columns, inplace=True, errors="ignore")

    return df

In [ ]:
# ================================================================================
# Main
# ================================================================================


In [ ]:
# Set output directory
output_file_path = DATA_DIR / "mapped"

In [133]:
# Set input file
excel_file_path = DATA_DIR / "raw" / "final_samobojstwa_2013_2022.csv"

In [134]:
df_raw = load_data(excel_file_path, is_excel=False)

In [135]:
df = df_raw.copy()

In [136]:
# Assuming df is your DataFrame and COLUMN_MAPPINGS is defined as shown
current_columns = df.columns.tolist()
mapped_columns = COLUMN_MAPPINGS.keys()

# Find columns in df that are not in COLUMN_MAPPINGS
unmapped_columns = [col for col in current_columns if col not in mapped_columns]

print("Columns in df that are not mentioned in COLUMN_MAPPINGS:")
print(unmapped_columns)

Columns in df that are not mentioned in COLUMN_MAPPINGS:
[]


In [137]:
df = map_columns(df, COLUMN_MAPPINGS)

In [138]:
# Checking column types
column_types = {}

for column in df.columns:
    # Get unique types in the column
    unique_types = set(df[column].dropna().apply(type))
    column_types[column] = unique_types

print("Unique types of values in each column:")
for col, types in column_types.items():
    print(f"{col}: {types}")

Unique types of values in each column:
ID: {<class 'str'>}
Date: {<class 'str'>}
AgeGroup: {<class 'int'>}
Gender: {<class 'int'>}
Marital: {<class 'int'>}
Education: {<class 'int'>}
WorkInfo: {<class 'int'>}
Income: {<class 'int'>}
Fatal: {<class 'int'>}
Place: {<class 'int'>}
Method: {<class 'int'>}
Context1: {<class 'int'>}
Context2: {<class 'str'>}
Context3: {<class 'float'>}
Context4: {<class 'float'>}
Substance: {<class 'int'>}
AbuseInfo1: {<class 'int'>}
AbuseInfo2: {<class 'int'>}


In [139]:
# Checking for unmapped values
unmapped_values = {}

for column in df.columns:
    if column in VALUE_MAPPINGS:
        # Get the mapping for the current column
        valid_values = set(
            map(str, VALUE_MAPPINGS[column].values())
        )  # Convert valid values to strings
        # Find unique values in the column that are not in the valid values
        column_values = set(
            map(str, df[column].dropna().unique())
        )  # Convert column values to strings
        unmapped = column_values - valid_values

        if unmapped:
            unmapped_values[column] = unmapped

print("Unmapped values in each column:")
for col, values in unmapped_values.items():
    print(f"{col}: {values}")

Unmapped values in each column:
AgeGroup: {'13', '10', '11', '7', '9', '4', '2', '5', '6', '8', '16', '0', '14', '12', '3', '15', '1'}
Gender: {'0', '1', '2'}
Marital: {'4', '5', '2', '6', '0', '3', '1'}
Education: {'7', '4', '5', '2', '6', '0', '3', '1'}
WorkInfo: {'7', '9', '4', '5', '6', '2', '8', '0', '3', '1'}
Income: {'4', '2', '0', '3', '1'}
Fatal: {'2'}
Place: {'13', '10', '11', '7', '9', '4', '2', '5', '6', '8', '0', '12', '14', '3', '15', '1'}
Method: {'13', '9', '4', '2', '8', '18', '17', '11', '7', '0', '12', '3', '15', '1', '10', '5', '6', '14', '16'}
Substance: {'13', '10', '11', '7', '1', '9', '4', '2', '8', '5', '6', '18', '0', '14', '12', '3', '17', '16'}
AbuseInfo1: {'7', '4', '5', '2', '6', '0', '3', '1'}
AbuseInfo2: {'0', '1', '2', '3'}


In [140]:
df = map_features(df, COLUMN_MAPPINGS, VALUE_MAPPINGS)

In [142]:
# Checking for unmapped values
unmapped_values = {}

for column in df.columns:
    if column in VALUE_MAPPINGS:
        # Get the mapping for the current column
        valid_values = set(
            map(str, VALUE_MAPPINGS[column].values())
        )  # Convert valid values to strings
        # Find unique values in the column that are not in the valid values
        column_values = set(
            map(str, df[column].dropna().unique())
        )  # Convert column values to strings
        unmapped = column_values - valid_values

        if unmapped:
            unmapped_values[column] = unmapped

print("Unmapped values in each column:")
for col, values in unmapped_values.items():
    print(f"{col}: {values}")

Unmapped values in each column:
Fatal: {'0.0', '1.0'}


In [145]:
# Checking for unmapped values
unmapped_values = {}

for column in df.columns:
    if column in VALUE_MAPPINGS:
        # Get the mapping for the current column
        valid_values = set(
            map(str, VALUE_MAPPINGS[column].values())
        )  # Convert valid values to strings
        # Find unique values in the column that are not in the valid values
        column_values = set(
            map(str, df[column].dropna().unique())
        )  # Convert column values to strings
        unmapped = column_values - valid_values

        if unmapped:
            unmapped_values[column] = unmapped

print("Unmapped values in each column:")
for col, values in unmapped_values.items():
    print(f"{col}: {values}")

Unmapped values in each column:
Fatal: {'0.0', '1.0'}


In [144]:
df = clean_data(df)